# ERM 424 · Module 3 · van Buuren Companion

Read a section of van Buuren, then run its cells here. The book's demonstrations are written in R. This notebook reproduces them in pandas, on the Module 2 claims file you already know, which holds no secrets by now. Plan on about 30 to 40 minutes alongside the reading. Nothing here is graded or submitted.

**Before anything else: File, then Save a copy in Drive.** Work in your copy.

Reading: van Buuren, S. (2018). [*Flexible imputation of missing data* (2nd ed.)](https://stefvanbuuren.name/fimd/). Chapman & Hall/CRC. Chapter 1: sections 1.1, 1.2, 1.3.1, 1.3.3, 1.3.7 and 1.3.8. Section 1.4 is optional. Section 1.1 sets the problem and has no demonstration to reproduce, so the cells below start at 1.2.

The book's figures use its own example data (the `airquality` data frame). Our figures show the same effects on our columns, so the numbers differ from the book's; the shapes are the point. The book marks observed data blue and imputed data red; our figures separate them by marker shape and line style instead, so they read without color.

Version 1.1, 2026-10-21.

In [ ]:
# Setup: run this cell first.
import sys, warnings
import numpy as np
import pandas as pd
EXPECTED = {"pandas": "2.2", "numpy": "2.1"}
for name, want in EXPECTED.items():
    have = __import__(name).__version__
    if not have.startswith(want):
        print(f"Note: {name} is {have}; this notebook was built on {want}.x. Most cells will still run.")
import os, time
import matplotlib.pyplot as plt
# Chart colors: the Okabe-Ito palette, readable with most forms of color-vision difference. Meaning never rides on color alone.
OKABE_ITO = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9", "#F0E442", "#000000"]
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=OKABE_ITO)
plt.rcParams["figure.dpi"] = 100

# Where the course files live. By default (Colab) the notebook reads them from the course repository on GitHub.
# If the course folders sit next to this notebook (a local copy of the repository), LOCAL becomes True and it reads them from disk.
REPO = "nybabot/ERM424"   # course repository on GitHub, as owner/name
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/claims/"
GENERATOR_URL = f"https://raw.githubusercontent.com/{REPO}/main/generators/claims_world.py"
LOCAL = False
DATA_DIR = "../../data/claims"   # notebooks/solutions, notebooks/student and docs/sync sit two folders below the course root
GEN_DIR = "../../generators"
for _cand in ["../../", "../", "./", "/content/"]:   # also works from qa/, from the root, or on a Colab VM with the folders under /content
    if os.path.isdir(_cand + "data/claims"):
        DATA_DIR, GEN_DIR = _cand + "data/claims", _cand + "generators"
        LOCAL = True
        break

def load(name, **kwargs):
    """Read one course CSV from the local data folder (LOCAL) or from the course repository (published)."""
    source = os.path.join(DATA_DIR, name) if LOCAL else BASE_URL + name
    return pd.read_csv(source, **kwargs)

print("pandas", pd.__version__, "| numpy", np.__version__, "| reading data from", os.path.abspath(DATA_DIR) if LOCAL else BASE_URL)


## Load and convert

Without this step, `dropna()` on the raw file would mislead you. Two lessons from Module 2 apply: a text label standing in for a missing value is invisible to `isna()`, and a column with no values at all makes every row look incomplete.

In [ ]:
# Load the Module 2 claims file as it arrives and count the disguised missing values in the three categorical columns.
claims = load("m2_claims.csv")
print(claims.shape[0], "rows,", claims.shape[1], "columns")
for col in ["vehicle_make", "report_channel", "region"]:
    print(f"{col}: isna() finds {claims[col].isna().sum()}, the label UNKNOWN is on {(claims[col] == 'UNKNOWN').sum()} rows")
print("vehicle_value true blanks:", claims["vehicle_value"].isna().sum())
print("notes column blank on", claims["notes"].isna().sum(), "of", len(claims), "rows")

In [ ]:
# Convert: UNKNOWN to a real blank in the three columns, drop the blank trailing column, parse the dates, then assert the counts.
df = claims.copy()
for col in ["vehicle_make", "report_channel", "region"]:
    df[col] = df[col].mask(df[col] == "UNKNOWN")
assert df["notes"].isna().all(), "the notes column is no longer blank"
df = df.drop(columns="notes")
for col in ["incident_date", "report_date"]:
    df[col] = pd.to_datetime(df[col], format="%Y-%m-%d")

expected = {"rows": (len(df), 1500), "columns": (df.shape[1], 17),
            "vehicle_make blanks": (int(df["vehicle_make"].isna().sum()), 196),
            "report_channel blanks": (int(df["report_channel"].isna().sum()), 64),
            "region blanks": (int(df["region"].isna().sum()), 18),
            "vehicle_value blanks": (int(df["vehicle_value"].isna().sum()), 144)}
for name, (found, want) in expected.items():
    assert found == want, f"{name}: expected {want}, found {found}. The data file has changed."
print("All counts match what Module 2 found. The data is as expected.")

## Section 1.2: Concepts of MCAR, MAR and MNAR

Three mechanisms for why a value is missing, in plain words:

- **MCAR** (missing completely at random): the chance a value is missing is the same for every row. Nothing in the data predicts it.
- **MAR** (missing at random): the chance depends on other columns you can see, but not on the missing value itself.
- **MNAR** (missing not at random): the chance depends on the missing value itself, or on something you did not record.

You cannot prove which one holds from the data alone. You can look for evidence. Here is what evidence for a MAR-style pattern looks like in pandas: the blank rate of `vehicle_value`, grouped by how the claim was reported.

In [ ]:
# Section 1.2: blank rate of vehicle_value by report channel, with group sizes. The 64 claims with an unknown channel form their own group.
by_channel = (df.assign(blank=df["vehicle_value"].isna())
                .groupby("report_channel", dropna=False)["blank"].agg(blank_rows="sum", rows="size"))
by_channel["blank_rate"] = (by_channel["blank_rows"] / by_channel["rows"]).round(3)
by_channel.index = by_channel.index.fillna("channel unknown")
by_channel = by_channel.sort_values("blank_rate", ascending=False)
display(by_channel)

In [ ]:
# Figure for section 1.2: blank rate of vehicle_value by report channel, with the group size under each label.
fig, ax = plt.subplots(figsize=(8, 3.8))
labels = [f"{name}\n(n={rows})" for name, rows in zip(by_channel.index, by_channel["rows"])]
ax.bar(labels, by_channel["blank_rate"] * 100)
ax.set_title("vehicle_value: share blank, by report channel")
ax.set_xlabel("report channel (number of claims)")
ax.set_ylabel("percent of vehicle_value blank")
plt.show()

Figure: bar chart of the blank rate of `vehicle_value` by report channel; Phone and Agent are about 17%, Email about 8%, and Mobile App, Web Portal and the unknown-channel group about 5%.

Phone and Agent claims are blank about three times as often as Mobile App and Web Portal claims, and each group holds hundreds of rows (the small groups are Email, 72, and the unknown channel, 64). That is the kind of pattern that points to MAR: the gaps are explained by another observed column. In this world, a claim reported through a digital channel carries the vehicle value from the valuation feed automatically, while Phone and Agent reports need it entered by hand.

Two cautions. This is evidence, not proof: another column you did not look at could explain the gaps better. And it cannot rule out MNAR, because nothing in the file shows whether high-value vehicles are the ones left blank.

## Section 1.3.1: Listwise deletion

Listwise deletion keeps only the rows with no missing value in any column. In pandas that is `dropna()`. The book shows it is unbiased when the data are MCAR, and it warns that it can lose most of the sample.

In [ ]:
# Section 1.3.1: dropna() on the raw file against dropna() after the conversion above.
print("rows in the raw file:", len(claims), "| rows left after dropna():", len(claims.dropna()))
print("rows in the converted file:", len(df), "| rows left after dropna():", len(df.dropna()),
      f"({len(df.dropna()) / len(df):.1%} survive)")

On the raw file, `dropna()` keeps no rows at all, because the `notes` column is blank on every row. After the conversion, about three quarters survive. Now watch what each additional column costs.

In [ ]:
# Section 1.3.1: the survival fraction as more columns must be complete.
required = ["vehicle_value", "vehicle_make", "report_channel", "region"]
rows = []
for k in range(1, len(required) + 1):
    kept = len(df.dropna(subset=required[:k]))
    rows.append({"must be complete": ", ".join(required[:k]), "rows kept": kept, "share kept": round(kept / len(df), 3)})
display(pd.DataFrame(rows))

In [ ]:
# Section 1.3.1: do the surviving rows look like all rows? Compare the report-channel mix and the mean total_amount.
complete = df.dropna()
mix = pd.DataFrame({"all rows": df["report_channel"].value_counts(normalize=True, dropna=False),
                    "complete rows": complete["report_channel"].value_counts(normalize=True)}).round(3)
mix.index = mix.index.fillna("channel unknown")
display(mix)
print("mean total_amount, all rows:", round(df["total_amount"].mean(), 2), "| complete rows:", round(complete["total_amount"].mean(), 2))

The survival fraction is the warning: each column you require to be complete removes more rows, and the rows that go are not a random sample. Here the loss is mild and the differences are small (the claims with an unknown channel vanish entirely, and Web Portal gains share), so deletion does little damage in this file. The point is to check every time. With a higher blank rate, or a stronger pattern, the same code would give a different answer, and `dropna()` will not tell you.

## Section 1.3.3: Mean imputation

Mean imputation fills every blank with the average of the observed values. The book's Figure 1.2 shows the result for the `Ozone` column of the `airquality` data: the imputed points line up at the mean, the distribution grows a spike there, the variance shrinks, and the relation with other variables weakens.

Here is the same experiment on `vehicle_value`, with `vehicle_year` as the other variable. **Correspondence:** the book's panels plot `Ozone` and `Ozone` against `Solar.R`; ours plot `vehicle_value`, and `vehicle_value` against `vehicle_year`. The book's `Ozone` is missing for about a quarter of rows; ours for 9.6%, so the effects here are smaller but have the same shape.

In [ ]:
# Section 1.3.3: mean imputation of vehicle_value, and what it did to the average, the spread and the relation with vehicle_year.
observed = df["vehicle_value"].dropna()
filled = df["vehicle_value"].fillna(observed.mean())
is_imputed = df["vehicle_value"].isna()
r_observed = df[["vehicle_value", "vehicle_year"]].dropna().corr().iloc[0, 1]
r_filled = pd.concat([filled, df["vehicle_year"]], axis=1).corr().iloc[0, 1]
print(f"mean:  observed {observed.mean():,.0f} | after filling {filled.mean():,.0f}")
print(f"std:   observed {observed.std():,.0f} | after filling {filled.std():,.0f} ({filled.std() / observed.std() - 1:+.1%})")
print(f"correlation with vehicle_year: observed rows {r_observed:.3f} | after filling {r_filled:.3f}")

In [ ]:
# Figure A (book Figure 1.2, right panel): vehicle_value against vehicle_year, observed points and mean-imputed points.
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(df.loc[~is_imputed, "vehicle_year"], df.loc[~is_imputed, "vehicle_value"], s=14, alpha=0.5,
           marker="o", label="observed")
ax.scatter(df.loc[is_imputed, "vehicle_year"], filled[is_imputed], s=28, marker="^", label="imputed with the mean")
ax.axhline(observed.mean(), color="black", linestyle="--", linewidth=1, label="observed mean")
ax.set_title("vehicle_value against vehicle_year, with mean imputation")
ax.set_xlabel("vehicle model year")
ax.set_ylabel("vehicle value (dollars)")
ax.legend()
plt.show()

Figure: scatter of vehicle value against model year; observed points (circles) rise with year, and the imputed points (triangles) sit on one horizontal line at the observed mean regardless of year.

In [ ]:
# Figure B (book Figure 1.2, left panel): the distribution of vehicle_value, observed against after mean imputation.
edges = np.arange(0, observed.max() + 2000, 2000)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(observed, bins=edges, density=True, histtype="step", linewidth=2, linestyle="-", label="observed")
ax.hist(filled, bins=edges, density=True, histtype="step", linewidth=2, linestyle="--", label="after mean imputation")
ax.set_title("vehicle_value: observed and after mean imputation")
ax.set_xlabel("vehicle value (dollars)")
ax.set_ylabel("density")
ax.legend()
plt.show()

Figure: overlay of the vehicle value distribution before and after mean imputation; the filled version adds a tall spike at the mean, and its spread is slightly smaller than the observed one.

What the two figures and the numbers show. The mean did not move, by construction. The standard deviation fell by about 5%, and the correlation with `vehicle_year` fell from 0.71 to 0.67: every imputed row ignores the year, so the relationship weakens. The spike at the mean is a value that no one observed, repeated 144 times. The book makes the same points with larger effects because more of its data are missing: for `Ozone`, the standard deviation falls from 33 to 28.7 and the correlation with `Solar.R` from 0.35 to 0.3. Mean imputation keeps the sample size and the average, and gives up the spread and the relationships.

## Section 1.3.7: Indicator method

The indicator method keeps every row. It fills each blank with a constant (the book uses zero) and adds a column that records whether the value was missing, then includes both in the model. The cell below builds the indicator beside a filled value, then fits a small regression of `vehicle_amount` on `vehicle_value` and vehicle age five ways, so you can see what the method preserves and what it distorts.

In [ ]:
# Section 1.3.7: the missingness indicator beside a filled value, and its blank rate by report channel.
indicator = pd.DataFrame({"vehicle_value": df["vehicle_value"],
                          "vehicle_value_missing": df["vehicle_value"].isna().astype(int),
                          "filled_with_zero": df["vehicle_value"].fillna(0),
                          "filled_with_mean": df["vehicle_value"].fillna(observed.mean())})
display(indicator[indicator["vehicle_value_missing"] == 1].head(3))
print("the indicator keeps the pattern from section 1.2:")
print(indicator["vehicle_value_missing"].groupby(df["report_channel"].fillna("channel unknown")).mean().round(3))

In [ ]:
# Section 1.3.7: coefficients of vehicle_amount on vehicle_value and vehicle age, five ways (least squares with numpy).
age = (2025 - df["vehicle_year"]).to_numpy(float)
y = df["vehicle_amount"].to_numpy(float)
value = df["vehicle_value"].to_numpy(float)
miss = df["vehicle_value"].isna().to_numpy()

def fit(columns, names, rows):
    """Least squares with an intercept; return the coefficients by name."""
    X = np.column_stack([np.ones(rows.sum())] + [col[rows] for col in columns])
    beta = np.linalg.lstsq(X, y[rows], rcond=None)[0]
    return dict(zip(["intercept"] + names, beta))

everyone = np.ones(len(y), dtype=bool)
mean_value = np.nanmean(value)
models = {
    "complete cases only": fit([value, age], ["vehicle_value", "age"], ~miss),
    "mean fill": fit([np.where(miss, mean_value, value), age], ["vehicle_value", "age"], everyone),
    "zero fill": fit([np.where(miss, 0, value), age], ["vehicle_value", "age"], everyone),
    "mean fill + indicator": fit([np.where(miss, mean_value, value), miss.astype(float), age],
                                 ["vehicle_value", "indicator", "age"], everyone),
    "zero fill + indicator": fit([np.where(miss, 0, value), miss.astype(float), age],
                                 ["vehicle_value", "indicator", "age"], everyone),
}
rows_used = {"complete cases only": int((~miss).sum())}
table = pd.DataFrame(models).T[["vehicle_value", "age", "indicator"]]
table.insert(0, "rows used", [rows_used.get(name, len(y)) for name in table.index])
display(table.round(3))

What the indicator method preserves: all 1,500 rows, and the fact of missingness itself, which the first table shows is tied to the report channel. What it distorts: the filled value is not data, and the coefficients depend on what you fill with. Compare the `age` column. Using complete cases only, an older vehicle goes with a slightly higher `vehicle_amount`; once the filled rows join the fit, the sign of the age coefficient reverses, and it moves again with a zero fill. The indicator's own coefficient is about 290 with a mean fill and about 3,900 with a zero fill, so it is an artifact of the fill value, not a finding.

Which row is right? The data cannot say from the fit alone. Complete cases are not a gold standard either, because the blanks are tied to the report channel. That is the book's conclusion about this method: it can bias the estimates even under MCAR, so it fails as a generic way to handle missing data.

## Section 1.3.8: Summary

The book closes its list of simple methods with Table 1.1, "Overview of assumptions made by ad-hoc methods". It is reproduced below as a DataFrame so you have it as a reference. The book's table is the source; the row labels and cells are the book's, and a dash in the book is written "No" here.

In [ ]:
# Section 1.3.8: the book's Table 1.1 as a DataFrame, one row per method (a dash in the book is written No).
table_1_1 = pd.DataFrame(
    [["Listwise", "MCAR", "MCAR", "MCAR", "Too large"],
     ["Pairwise", "MCAR", "MCAR", "MCAR", "Complicated"],
     ["Mean", "MCAR", "No", "No", "Too small"],
     ["Regression", "MAR", "MAR", "No", "Too small"],
     ["Stochastic", "MAR", "MAR", "MAR", "Too small"],
     ["LOCF", "No", "No", "No", "Too small"],
     ["Indicator", "No", "No", "No", "Too small"]],
    columns=["Method", "Unbiased: mean", "Unbiased: regression weight", "Unbiased: correlation", "Standard error"]).set_index("Method")
display(table_1_1)

How to read a row. A cell that names a mechanism means the method gives unbiased estimates only if the data are of that kind: Listwise needs MCAR for all three estimates. A "No" means the method cannot give unbiased estimates, even under MCAR. The regression weight column is for a model with the incomplete variable as the dependent variable. The last column is about the standard error: Listwise gives one that is correct for the complete cases but too large for the whole dataset, Pairwise is complicated, and every single-imputation method gives one that is too small, because the filled values are treated as if they were observed. The book's answer to the "too small" column is multiple imputation, its section 1.4.

Seeing these results in your own data (the spike, the weaker correlation, the sign change) is the reason the table is worth keeping.

## Where this leaves you

Each simple treatment destroys something: deletion discards data, mean imputation invents certainty, and flagging distorts the estimates it touches. None of them is a safe default, and each is sometimes the right choice. The professional response to imperfect options is a decision log: name the issue, the mechanism you suspect and your evidence, the treatment, the alternatives you rejected, and the information you lost.

The principled method has a name, **multiple imputation**: fill each gap several times with plausible values, analyze each completed version, and pool the results so the answer carries its own uncertainty. It is deferred, not skipped. Its outputs are statements about uncertainty, which are readable only once Module 6's inference tools arrive, so we return to it there. If you are curious now, section 1.4 of the book is short and optional; nothing this week depends on it.